# Suite CHG — Changes: what a transaction changed, observed at its commit, as a journal

A journal, a viewer or a layout engine needs to know exactly what a transaction changed, which its expression, a query that writes, does not say ([CAP-270: a transaction's changes observed at its commit](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-270)). `Changes.observe(store, observer)` gives `observer` a journal of each transaction that commits having changed something, a `Changes.Journal` of that one transaction, and returns a function that ends it. The journal is written in the journal language, the `Changes` schemas every store holds ([the journals proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/JOURNALS.md#the-changes-schemas-proposed)): a transaction's `id`, the uid `Provenance` records it by; its symbols, each bound to an object held before, by its reference, or to a new object, by its schema's; then its changes, in order, relation entries removed and inserted and properties set and cleared, net of what the transaction undid within itself, a counted object written by its value where an entry links it. A journal is data, written, read and compared as any value is. `Changes.apply(txn, journal)` applies it: to the store as it was, it makes the same changes; applied again, none.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Changes, Comparison, Plain, Proxies
from mbse.Schemas.Framework.Errors import DecodeError
from support import raises


def crm(txn):
    """crm.Person, owned by the singleton book by key, with a name and a home, who knows persons since a year, and
    pins persons through no adjacency by name; crm.Note, counted, about a person."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Home": S.OfObject, "Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation,
                  "Book": S.OfObject, "Note": S.OfObject, "About": S.OfRelation, "Pinned": S.OfRelation})
        .spec([
            lambda _: _.Home.name("crm.Home").properties({"city": S.OfStr, "zip": S.OfStr}),
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")])
            .properties({"since": S.OfInt}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr, "home": _.Home}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
                lambda t: t.relation([_.Pinned]).me("pinner"),  # no adjacency to crm.Pinned has a name
                lambda t: t.relation([_.Pinned]).me("pinned"),
            ]),
            lambda _: _.Pinned.name("crm.Pinned").links([lambda t: t.name("pinner"), lambda t: t.name("pinned")]),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")])
            .properties({"key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}).adjacencies({
                "about": lambda t: t.relation([_.About]).me("note")}),
            lambda _: _.About.name("crm.About").links([lambda t: t.name("note"), lambda t: t.name("person")]),
        ])]))])


store = B.Store()
store.transaction(crm)
Person, Book = store.registered("crm.Person"), store.registered("crm.Book")


def contact(key):
    """The contact the book holds by `key`, within the transaction open: found again by navigating from the singleton."""
    return store.singleton("crm.TheBook").contacts({"key": key})


def short(path):
    """A path, short: its singleton, then each key it passes, `crm.TheBook/ada`; a schema's, its name."""
    parts, at = [path["singleton"]], path.get("next", {})
    while "entry" in at:
        parts += [str(v) for v in at["entry"].get("properties", {}).values()]
        at = at["entry"].get("next", {})
    return ".".join(parts[1:]) if parts[0] == "." else "/".join(parts)


def plain(journal, source=None):
    """A journal of `source`'s, the crm store unless given, as plain data."""
    source = source or store
    return Plain.ToPlain(source)(source.registered("Changes.Journal"), journal)


def shown(journal, source=None):
    """A journal of one transaction of `source`, the crm store unless given, short: each symbol, as what it is bound
    to, then each change, in order; a counted object inline, by its values."""
    (transaction,) = plain(journal, source)

    def linked(link):
        if "def" in link:
            values = " ".join(f"{m['as_property']['name']}={json.dumps(m['as_property']['mutations'][0]['set']['value'])}"
                              for m in link["def"]["mutations"])
            return f"({link['def']['schema']} {values})"
        return link["reference"]["symbol"]

    lines = [f"{name} = {short(bound['reference']['path'])}" if "reference" in bound
             else f"{name} = new {short(bound['schema']['path'])}" for name, bound in transaction["symbols"].items()]
    for change in transaction["changes"]:
        if "property" in change:
            p = change["property"]
            (verb,) = p["mutations"]
            lines.append(f"{p['holder']}.{p['name']} = " + (json.dumps(verb["set"]["value"]) if "set" in verb else "cleared"))
            continue
        e = change["entry"]
        parts = [f"{n}={linked(link)}" for n, link in e.get("links", {}).items()] + [
            f"{n}={json.dumps(v)}" for n, v in e.get("properties", {}).items()]
        lines.append(f"{'+' if e['verb'] == 'insert' else '-'} {linked(e['holder'])}.{e['adjacency']}({', '.join(parts)})")
    return lines


def people(*names, source=None):
    """A transaction of `source`'s, the crm store unless given, adding persons to the book, each keyed by its name."""
    source = source or store
    book, person = source.registered("crm.Book"), source.registered("crm.Person")
    return lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"book": book, **{n: person for n in names}})
        .spec([lambda _: _.book.contacts({n: (lambda e, n=n: e.contact(getattr(_, n))) for n in names}),
               *[(lambda _, n=n: getattr(_, n).name(n)) for n in names]])]))])


def applied(target, journal, source=None):
    """Applies in `target` a journal of `source`'s, the crm store unless given, carried as its plain form."""
    carried = json.loads(JSON.dumps(plain(journal, source)))
    read = Plain.FromPlain(target)(target.registered("Changes.Journal"), carried)
    target.transaction(lambda txn: Changes.apply(txn, read))


data = lambda s: JSON.dumps(Plain.ToPlain(s).Snapshot()[1])  # its data, its schemas aside
log = []
end = Changes.observe(store, log.append)

<a id="CHG-01"></a>
## CHG-01 · Each transaction that commits having changed something is given to the observer as a journal of that one transaction, after its commit, a value that outlives it; one that changes nothing is not

In [ ]:
store.transaction(people("ada", "bob"))
assert shown(log[-1]) == ["o0 = crm.TheBook", "o1 = new crm.Person", "o2 = new crm.Person",
                          '+ o0.contacts(contact=o1, key="ada")', '+ o0.contacts(contact=o2, key="bob")',
                          'o1.name = "ada"', 'o2.name = "bob"']
assert log[-1].as_schema() is store.registered("Changes.Journal") and len(log[-1]) == 1  # schema data, read outside
count = len(log)
store.transaction(people("ada"))  # changes nothing: the spec holds
assert len(log) == count

<a id="CHG-02"></a>
## CHG-02 · A transaction that raises, its writes dropped, is observed by none

In [ ]:
count = len(log)
with raises(RuntimeError, match="refused"):
    with store.transaction():
        store.crm.Person(contact("ada"), lambda p: p.name("Zed"))
        raise RuntimeError("refused")
with store.transaction():
    assert len(log) == count and contact("ada").name.get() == "ada"

<a id="CHG-03"></a>
## CHG-03 · What a transaction undid within itself is no change: an entry inserted and removed, a value set and set back

In [ ]:
count = len(log)
with store.transaction():
    ada, bob = contact("ada"), contact("bob")
    store.crm.Person(ada, lambda p: p.knows(lambda a: a.insert(lambda e: e.known(bob).since(2001))))
    store.crm.Person(ada, lambda p: p.knows(lambda a: a.remove(lambda e: e.known(bob))))
    store.crm.Person(ada, lambda p: p.name("Ada"))
    store.crm.Person(ada, lambda p: p.name("ada"))
assert len(log) == count

<a id="CHG-04"></a>
## CHG-04 · Removing an owning entry deletes what it owned: the journal removes that entry by its owner's key, naming nothing deleted, and what goes with it follows

In [ ]:
with store.transaction():
    store.crm.Person(contact("ada"), lambda p: p.knows(lambda a: a.insert(lambda e: e.known(contact("bob")).since(2001))))
assert shown(log[-1]) == ["o0 = crm.TheBook/bob", "o1 = crm.TheBook/ada", "+ o1.knows(known=o0, since=2001)"]
with store.transaction():
    store.crm.Book(store.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("bob"))))
assert shown(log[-1]) == ["o0 = crm.TheBook", '- o0.contacts(key="bob")']

<a id="CHG-05"></a>
## CHG-05 · A property changed is set to its value, written whole and bare by its schema, a value object's part included, or cleared

In [ ]:
with store.transaction():
    store.crm.Person(contact("ada"), lambda p: p.home(lambda h: h.city("Paris").zip("75001")))
with store.transaction():
    store.crm.Person(contact("ada"), lambda p: p.home(lambda h: h.city("Lyon")))
assert shown(log[-1]) == ["o0 = crm.TheBook/ada", 'o0.home = {"city": "Lyon", "zip": "75001"}']
with store.transaction():
    store.crm.Person(contact("ada"), lambda p: p.name(None))
assert shown(log[-1]) == ["o0 = crm.TheBook/ada", "o0.name = cleared"]

<a id="CHG-06"></a>
## CHG-06 · A counted object, immutable, is written by its value, inline, in the entry change that links it, never bound to a symbol: found by its value where the journal is applied, or made

In [ ]:
with store.transaction():
    store.crm.Note(lambda n: n.text("hello").about(lambda a: a.insert(lambda e: e.person(contact("ada")))))
assert shown(log[-1]) == ["o0 = crm.TheBook/ada", '+ (crm.Note text="hello").about(person=o0)']
(change,) = plain(log[-1])[0]["changes"]
assert change["entry"]["holder"] == {"def": {"schema": "crm.Note", "mutations": [
    {"as_property": {"name": "text", "mutations": [{"set": {"value": "hello"}}]}}]}}  # a link set, defined where linked

<a id="CHG-07"></a>
## CHG-07 · Each journal, written and read as plain data and applied in order to a store as the first was (`Changes.apply`), makes it the same, applied twice the same again; once the observing ends, none is given

In [ ]:
replica = B.Store()
replica.transaction(crm)
for journal in log:
    for _ in range(2):
        applied(replica, journal)
assert data(replica) == data(store)
end()
end()  # ending it again changes nothing
count = len(log)
store.transaction(people("eve"))
assert len(log) == count

<a id="CHG-08"></a>
## CHG-08 · An entry that no object it links names an adjacency to cannot be written in the journal: while observed, its transaction is refused, naming its relation

In [ ]:
pinned = store.registered("crm.Pinned")
stop = Changes.observe(store, lambda journal: None)
with raises(ValueError, match="no object an entry of 'crm.Pinned' links has an adjacency to it by name"):
    with store.transaction():  # only the framework's own writes reach such a relation
        ada = contact("ada")
        Proxies._add_entry(pinned, Proxies._Entry(pinned, {"pinner": ada, "pinned": ada}, {}))
stop()
with store.transaction():
    assert not Proxies._linking(pinned, "pinner", contact("ada"))

<a id="CHG-09"></a>
## CHG-09 · Objects of many schemas owned through one relation: deleting one is written by its owner's key alone, with what it owned and every entry linking it following, applied twice the same; an owner with no adjacency by name to the relation cannot write it

In [ ]:
def org(txn):
    """org.Team and org.Member, each owned through org.Owns, by name, as is org.Badge, a member's; members are buddies;
    org.Key is owned by the singleton vault through no adjacency by name."""
    S = txn.Schemas

    def owned(t, _):
        return t.name("owner").relation([_.Owns]).me("owned").owner(True)

    return txn.uid("org/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Owns": S.OfRelation, "Root": S.OfObject, "Team": S.OfObject, "Member": S.OfObject,
                  "Badge": S.OfObject, "Buddy": S.OfRelation, "Holds": S.OfRelation, "Vault": S.OfObject,
                  "Key": S.OfObject})
        .spec([
            lambda _: _.Owns.name("org.Owns").links([lambda t: t.name("owner"), lambda t: t.name("owned")])
            .properties({"name": S.OfStr}).uniques([["owner", "name"], ["owned"]]),
            lambda _: _.Buddy.name("org.Buddy").links([lambda t: t.name("a"), lambda t: t.name("b")]),
            lambda _: _.Holds.name("org.Holds").links([lambda t: t.name("vault"), lambda t: t.name("key")])
            .properties({"name": S.OfStr}).uniques([["vault", "name"], ["key"]]),
            lambda _: _.Root.name("org.Root").ref(True).singleton("org.TheRoot").adjacencies({
                "owns": lambda t: t.relation([_.Owns]).me("owner")}),
            lambda _: _.Team.name("org.Team").ref(True).adjacencies([
                lambda t: owned(t, _), lambda t: t.name("owns").relation([_.Owns]).me("owner")]),
            lambda _: _.Member.name("org.Member").ref(True).adjacencies([
                lambda t: owned(t, _), lambda t: t.name("owns").relation([_.Owns]).me("owner"),
                lambda t: t.name("buddies").relation([_.Buddy]).me("a"),
                lambda t: t.relation([_.Buddy]).me("b")]),
            lambda _: _.Badge.name("org.Badge").ref(True).properties({"code": S.OfStr}).adjacencies([
                lambda t: owned(t, _)]),
            lambda _: _.Vault.name("org.Vault").ref(True).singleton("org.TheVault").adjacencies([
                lambda t: t.relation([_.Holds]).me("vault")]),  # no adjacency by name
            lambda _: _.Key.name("org.Key").ref(True).adjacencies([
                lambda t: t.name("vault").relation([_.Holds]).me("key").owner(True)]),
        ])]))])


first, second = B.Store(), B.Store()
for s in (first, second):
    s.transaction(org)
journals = []
Changes.observe(first, journals.append)
with first.transaction():
    root = first.singleton("org.TheRoot")
    team = first.org.Team(lambda t: t.owner(lambda a: a.insert(lambda e: e.name("core").owner(root))))
    ann = first.org.Member(lambda m: m.owner(lambda a: a.insert(lambda e: e.name("ann").owner(team))))
    ben = first.org.Member(lambda m: m.owner(lambda a: a.insert(lambda e: e.name("ben").owner(team)))
                           .buddies(lambda a: a.insert(lambda e: e.b(ann))))
    first.org.Badge(lambda b: b.owner(lambda a: a.insert(lambda e: e.name("pass").owner(ann))).code("A1"))
team = lambda: first.singleton("org.TheRoot").owns({"name": "core"})  # found again within each transaction
member = lambda name: team().owns({"name": name})
with first.transaction():
    first.org.Member(member("ben"), lambda m: m.buddies(lambda a: a.remove(lambda e: e.b(member("ann")))))
assert shown(journals[-1], first) == ["o0 = org.TheRoot/core/ann", "o1 = org.TheRoot/core/ben", "- o1.buddies(b=o0)"]
with first.transaction():
    first.org.Member(member("ben"), lambda m: m.buddies(lambda a: a.insert(lambda e: e.b(member("ann")))))
with first.transaction():
    first.org.Team(team(), lambda t: t.owns(lambda a: a.remove(lambda e: e.name("ann"))))
assert shown(journals[-1], first) == ["o0 = org.TheRoot/core", '- o0.owns(name="ann")']
for journal in journals:
    for _ in range(2):
        applied(second, journal, first)
assert data(second) == data(first)
with first.transaction():
    assert [e.name.get() for e in team().owns] == ["ben"] and not list(member("ben").buddies)
with first.transaction():
    first.org.Key(lambda k: k.vault(lambda a: a.insert(lambda e: e.name("k1").vault(first.singleton("org.TheVault")))))
with raises(ValueError, match="no owner of an entry of 'org.Holds' has an adjacency to it by name"):
    with first.transaction():
        (held,) = Proxies._linking(first.registered("org.Holds"), "vault", first.singleton("org.TheVault"))  # no adjacency by name
        key = held.key
        first.org.Key(key, lambda k: k.vault(lambda a: a.clear()))

<a id="CHG-10"></a>
## CHG-10 · The journal language's schemas are every store's own, as the reference schemas are: `Changes.Journal`, `Changes.Transaction`, `Changes.Symbol`, `Changes.Change`, `Changes.OfEntry` and `Changes.OfProperty`, value schemas the store was made holding; installed again, nothing changes

In [ ]:
fresh = B.Store()
names = ["Changes.Journal", "Changes.Transaction", "Changes.Symbol", "Changes.Change", "Changes.OfEntry",
         "Changes.OfProperty"]
assert all(fresh.framework(fresh.registered(name)) and not getattr(fresh.registered(name), "ref", False)
           for name in names)
assert fresh.registered("Changes.Journal") is not store.registered("Changes.Journal")  # each store's own
before = [fresh.registered(name) for name in names]
fresh.transaction(Changes.install)  # found by their names
assert [fresh.registered(name) for name in names] == before
assert list(fresh.registered("Changes.Transaction").properties) == ["id", "symbols", "changes"]
assert list(fresh.registered("Changes.OfEntry").properties) == ["holder", "adjacency", "verb", "links", "properties"]
assert list(fresh.registered("Changes.OfProperty").properties) == ["holder", "name", "mutations"]

<a id="CHG-11"></a>
## CHG-11 · A transaction's journal has an id, unique: the uid `Provenance` records the transaction by, its expression's where it gives one, else one the store makes, whether or not `Provenance` is installed

In [ ]:
from mbse.Schemas.Framework import Provenance

ids = []
stop = Changes.observe(store, lambda journal: ids.append(plain(journal)[0]["id"]))
store.transaction(lambda t: people("fay")(t.uid("add fay")))
with store.transaction():
    store.crm.Person(contact("fay"), lambda p: p.name("Fay"))
with store.transaction():
    store.crm.Person(contact("fay"), lambda p: p.name("fay"))
stop()
assert ids[0] == "add fay" and len(set(ids)) == 3 and all(isinstance(i, str) and i for i in ids)
recorded = B.Store()
recorded.transaction(crm)
recorded.transaction(Provenance.install)
recorded.transaction(lambda t: Provenance.extend(t, [recorded.registered("crm.Person")]))
seen = []
Changes.observe(recorded, lambda journal: seen.append(plain(journal, recorded)[0]["id"]))
with recorded.transaction():
    book = recorded.singleton("crm.TheBook")
    recorded.crm.Book(book, lambda b: b.contacts({"gus": lambda e: e.contact(lambda p: p.name("gus"))}))
with recorded.transaction():  # the transaction recorded by the id its journal has
    assert seen[-1] in [str(e.uid) for e in recorded.singleton("Provenance.Transactions").transactions]

<a id="CHG-12"></a>
## CHG-12 · A journal is data, with no store of its own: its transactions read by iterating it, written and read as plain data the same, and compared as any value is; whole transactions taken from it, in order, are a journal, which applies alone, and applies again changing nothing; a journal that says what no journal can is refused

In [ ]:
source, target = B.Store(), B.Store()
for s in (source, target):
    s.transaction(crm)
taken = []
stop = Changes.observe(source, taken.append)
source.transaction(people("ann", "cal", source=source))
with source.transaction():
    book = source.singleton("crm.TheBook")
    source.crm.Person(book.contacts({"key": "ann"}), lambda p: p.knows(lambda a: a.insert(
        lambda e: e.known(book.contacts({"key": "cal"})).since(1999))))
with source.transaction():
    source.crm.Person(source.singleton("crm.TheBook").contacts({"key": "cal"}), lambda p: p.name("Cal"))
stop()
whole = [transaction for journal in taken for transaction in plain(journal, source)]  # one journal of all three
Journal = target.registered("Changes.Journal")
journal = Plain.FromPlain(target)(Journal, json.loads(JSON.dumps(whole)))
read = []
journal.as_entries(lambda items: [t.as_object(lambda o: read.append(o.id.get()) or o) for _, t in items])
assert read == [t["id"] for t in whole] and len(journal) == 3  # iterated, in order
assert Plain.ToPlain(target)(Journal, journal) == whole  # written back the same
again = Plain.FromPlain(target)(Journal, whole)
assert Comparison.OfContainer(Journal, journal).compare(Comparison.OfContainer(Journal, again)) == 0
part = Plain.FromPlain(target)(Journal, whole[:2])  # its first two transactions: a journal
assert Comparison.OfContainer(Journal, journal).compare(Comparison.OfContainer(Journal, part)) is None
for _ in range(2):
    target.transaction(lambda txn: Changes.apply(txn, part))
with target.transaction():
    held = target.singleton("crm.TheBook")
    assert [(str(e.key), str(e.contact.name)) for e in held.contacts] == [("ann", "ann"), ("cal", "cal")]
    assert [(str(e.known.name), e.since.get()) for e in held.contacts({"key": "ann"}).knows] == [("cal", 1999)]
target.transaction(lambda txn: Changes.apply(txn, Plain.FromPlain(target)(Journal, whole[2:])))
assert data(target) == data(source)
target.transaction(lambda txn: Changes.apply(txn, Plain.FromPlain(target)(Journal, [])))  # nothing to apply
assert data(target) == data(source)
symbols = {"c0": {"reference": {"path": {"singleton": "crm.TheBook", "next": {"entry": {
    "i": "contacts", "o": "book", "properties": {"key": "ann"}, "next": {"target": {}}}}}}}}  # a symbol named as a counted
note = {"def": {"schema": "crm.Note", "mutations": [  # object would be: the counted object's is another
    {"as_property": {"name": "text", "mutations": [{"set": {"value": "met"}}]}},
    {"as_adjacency": {"name": "about", "mutations": [{"insert": {"entries": [{"mutations": [
        {"as_link": {"name": "person", "mutations": [{"set": {"reference": {"symbol": "c0"}}}]}}]}]}}]}}]}}
cal = {"reference": {"path": {"singleton": "crm.TheBook", "next": {"entry": {
    "i": "contacts", "o": "book", "properties": {"key": "cal"}, "next": {"target": {}}}}}}}
noted = [{"id": "n1"}, {"id": "n2", "symbols": {**symbols, "o1": cal}, "changes": [{"entry": {
    "holder": note, "adjacency": "about", "verb": "insert", "links": {"person": {"reference": {"symbol": "c0"}}}}},
    {"entry": {"holder": note, "adjacency": "about", "verb": "insert", "links": {"person": {"reference": {"symbol": "o1"}}}}}]}]
for _ in range(2):
    target.transaction(lambda txn: Changes.apply(txn, Plain.FromPlain(target)(Journal, noted)))
with target.transaction():
    ann = target.singleton("crm.TheBook").contacts({"key": "ann"})
    (about,) = Proxies._linking(target.registered("crm.About"), "person", ann)
    (also,) = Proxies._linking(target.registered("crm.About"), "person", target.singleton("crm.TheBook").contacts(
        {"key": "cal"}))
    assert about.note.text.get() == "met" and also.note is about.note  # one note, of one value, found again
for bad, why in (({"verb": "clear"}, "an entry's change is an insert or a remove, not 'clear'"),
                 ({"links": {"person": {"reference": {"path": symbols["c0"]["reference"]["path"]}}}},
                  "a change links an object by a symbol of its transaction, or a counted object by its value")):
    wrong = json.loads(json.dumps(noted))
    wrong[1]["changes"][0]["entry"].update(bad)
    with raises(ValueError, match=why):
        target.transaction(lambda txn: Changes.apply(txn, Plain.FromPlain(target)(Journal, wrong)))
with raises(ValueError, match="a journal, as plain data, is a list of its transactions"):
    target._arrive({"id": "n3"})

<a id="CHG-13"></a>
## CHG-13 · A value in a change is written by the schema of the property it is the value of, bare, through any of mbse-schemas' verbs of the value's kind, a value of a schema with no name of its own among them, and marked with its schema only where that schema cannot tell it, a refinement's; read back the same; one of a property the holder's schema lacks is a value of any schema, tagged, and refused bare

In [ ]:
def val(txn):
    """val.Thing, the singleton val.TheThing, with tags, a list of text, a point or a refinement of one, a number or a
    text, and a count; val.Mark, counted, marks a thing by a weight."""
    S = txn.Schemas
    return txn.uid("val/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Point": S.OfObject, "Point3": S.OfObject, "Thing": S.OfObject, "Mark": S.OfObject,
                  "Marks": S.OfRelation})
        .spec([
            lambda _: _.Point.name("val.Point").properties({"x": S.OfInt, "y": S.OfInt}),
            lambda _: _.Point3.name("val.Point3").refines([_.Point]).properties({"z": S.OfInt}),
            lambda _: _.Marks.name("val.Marks").links([lambda t: t.name("mark"), lambda t: t.name("thing")])
            .properties({"weight": S.OfInt}),
            lambda _: _.Thing.name("val.Thing").ref(True).singleton("val.TheThing").properties({
                "tags": S.OfContainer(lambda c: c.index([S.OfInt]).element([S.OfStr])),
                "at": _.Point,
                "either": S.OfUnion(lambda u: u.branches([lambda t: t.name("n").schema([S.OfInt]),
                                                         lambda t: t.name("s").schema([S.OfStr])])),
                "count": S.OfInt,
            }).adjacencies({"marks": lambda t: t.relation([_.Marks]).me("thing")}),
            lambda _: _.Mark.name("val.Mark").ref(True).properties({"label": S.OfStr}).adjacencies({
                "on": lambda t: t.relation([_.Marks]).me("mark")}),
        ])]))])


values = B.Store()
values.transaction(val)
thing = {"reference": {"path": {"singleton": "val.TheThing", "next": {"target": {}}}}}
verbs = [{"id": "v1", "symbols": {"x": thing}, "changes": [
    {"property": {"holder": "x", "name": "tags", "mutations": [{"set": {"value": ["a", "b"]}}, {"as_any": {"mutations": [
        {"as_container": {"mutations": [{"insert": {"slice": {"index": 1, "items": ["c"]}}}, {"remove": {"index": 0, "count": 1}},
                                        {"append": {"items": ["d"]}}, {"prepend": {"items": ["e"]}},
                                        {"as_items": {"mutations": [{"set": {"value": "f"}}]}}, {"clear": {}}]}}]}}]}},
    {"property": {"holder": "x", "name": "at", "mutations": [{"set": {"value": {"$schema": "val.Point3", "x": 1, "y": 2, "z": 3}}},
        {"as_any": {"mutations": [{"as_object": {"mutations": [{"name": "x", "mutations": [{"set": {"value": 5}}]}]}}]}}]}},
    {"property": {"holder": "x", "name": "either", "mutations": [{"set": {"value": {"s": "one"}}}, {"as_any": {"mutations": [
        {"as_union": {"mutations": [{"name": "n", "mutations": [{"set": {"value": 4}}]}]}}]}}]}},
    {"property": {"holder": "x", "name": "count", "mutations": [{"as_any": {"mutations": [
        {"as_scalar": {"mutations": [{"set": {"value": 3}}, {"clear": {}}]}}, {"set": {"value": 2}}, {"clear": {}}]}},
        {"clear": {}}]}},
    {"entry": {"holder": {"reference": {"symbol": "x"}}, "adjacency": "marks", "verb": "insert", "links": {"mark": {"def": {
        "schema": "val.Mark", "mutations": [{"as_property": {"name": "label", "mutations": [{"set": {"value": "m"}}]}},
            {"as_adjacency": {"name": "on", "mutations": [{"insert": {"entries": [{"mutations": [
                {"as_link": {"name": "thing", "mutations": [{"set": {"reference": {"symbol": "x"}}}]}},
                {"as_property": {"name": "weight", "mutations": [{"set": {"value": 8}}]}}]}]}}]}}]}}},
        "properties": {"weight": 7}}},
]}]
Values = values.registered("Changes.Journal")
assert Plain.ToPlain(values)(Values, Plain.FromPlain(values)(Values, verbs)) == verbs  # bare by its schema, the same
assert JSON.dumps(Plain.ToPlain(values)(Values, Plain.FromPlain(values)(Values, verbs))) == JSON.dumps(verbs)
unknown = [{"id": "v2", "symbols": {"x": thing}, "changes": [
    {"property": {"holder": "x", "name": "size", "mutations": [{"set": {"value": {"int": 9}}}]}},
    {"entry": {"holder": {"reference": {"symbol": "x"}}, "adjacency": "marks", "verb": "insert",
               "properties": {"height": {"int": 2}}}}]}]
assert Plain.ToPlain(values)(Values, Plain.FromPlain(values)(Values, unknown)) == unknown  # tagged: no schema says
unknown[0]["changes"][0]["property"]["mutations"][0]["set"]["value"] = 9
with raises(DecodeError, match="a value of any schema is {schema: value}"):
    Plain.FromPlain(values)(Values, unknown)

<a id="CHG-14"></a>
## CHG-14 · `store.follow(follower)` gives each transaction the store commits as a journal of that one transaction, as plain data, the journal `Changes.observe` gives; a store above applies it to what it holds, under its id (suite BLW); what a store commits for a store above, it gives back as a mutation transaction, not a journal

In [ ]:
below = B.Store()
below.transaction(crm)
given, followed = [], []
Changes.observe(below, given.append)
stop = below.follow(followed.append)
below.transaction(people("hal", source=below))
stop()
assert followed == [plain(given[0], below)] and followed[0][0]["changes"][0]["entry"]["adjacency"] == "contacts"
above = B.Store(below)
above.transaction(crm)
with above.transaction():
    hal = above.singleton("crm.TheBook").contacts({"key": "hal"})  # present above
above_given = []
Changes.observe(above, above_given.append)
with below.transaction():
    below.crm.Person(below.singleton("crm.TheBook").contacts({"key": "hal"}), lambda p: p.name("Hal"))
with above.transaction():
    assert above.singleton("crm.TheBook").contacts({"key": "hal"}).name.get() == "Hal"
(arrived,) = above_given
assert shown(arrived, above) == ["o0 = crm.TheBook/hal", 'o0.name = "Hal"']
assert plain(arrived, above)[0]["id"] == plain(given[-1], below)[0]["id"]  # under the id it arrived with
unnamed = [{k: v for k, v in transaction.items() if k != "id"} for transaction in plain(given[-1], below)]
above._arrive(json.loads(JSON.dumps(unnamed)))  # a journal whose transaction has no id, as a follow gives none: held,
assert len(above_given) == 1  # it changes nothing
alone = B.Store()  # a store below that none follows gives back what it committed, a mutation transaction, not a journal
alone.transaction(crm)
committed = alone.commit(Plain.ToPlain(alone).Transaction(alone.Mutations.OfTransaction(people("ivy", source=alone))))
assert list(committed) == ["mutations"] and "upsert" in committed["mutations"][0]